In [3]:
import numpy as np 
import matplotlib.pyplot as plt
from pydrake.systems.framework import LeafSystem, DiagramBuilder 
from pydrake.systems.analysis import Simulator
from pydrake.systems.primitives import ConstantVectorSource, LogVectorOutput

In [2]:
class Pendulum(LeafSystem):
    def __init__(self, g=9.81, l=1.0, b=0.1):
        super().__init__()
        self.g, self.l, self.b = g, l, b
        self.DeclareContinuousState(2)
        self.DeclareVectorInputPort('u', 1)
        self.DeclareVectorOutputPort('x', 2, self.CopyState) 
    
    def DoCalcTimeDerivatives(self, context, derivatives):
        q, dq = context.get_continuous_state_vector().CopyToVector()
        u = self.get_input_port(0).Eval(context)[0]
        ddq = - (self.g / self.l) * np.sin(q) - self.b *dq + u 
        derivatives.get_mutable_vector().SetFromVector([q, dq])
    
    def CopyState(self, context, output):
        output.SetFromVector(context.get_continuous_state_vector().CopyToVector())

In [8]:
builder = DiagramBuilder()
pendulum = builder.AddSystem(Pendulum())
torque = builder.AddSystem(ConstantVectorSource([0.0]))
builder.Connect(torque.get_output_port(), pendulum.get_input_port())
logger = LogVectorOutput(pendulum.get_output_port(), builder)
diagram = builder.Build()

simulator = Simulator(diagram)
context = simulator.get_mutable_context() 
pendulum_context = pendulum.GetMyMutableContextFromRoot(context)
pendulum_context.SetContinuousState([np.pi / 2, 0.0])

simulator.AdvanceTo(10.0)

log = logger.FindLog(context)

t, x = log.sample_times(), log.data()
print(t)
print(x)
for time in [0, 1, 2, 5, 10]:
    i = np.argmin(abs(t - time))
    print(f"t={t[i]:5.2f}s  theta={x[0, i]:+.3f} rad  theta_dot={x[1, i]:+.3f} rad/s")

[0.00000000e+00 1.00000000e-04 6.00000000e-04 ... 9.99637294e+00
 9.99884374e+00 1.00000000e+01]
[[1.57079633e+00 1.57095341e+00 1.57173909e+00 ... 3.44733910e+04
  3.45586728e+04 3.45986549e+04]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]]
t= 0.00s  theta=+1.571 rad  theta_dot=+0.000 rad/s
t= 0.99s  theta=+4.230 rad  theta_dot=+0.000 rad/s
t= 1.99s  theta=+11.533 rad  theta_dot=+0.000 rad/s
t= 5.00s  theta=+233.616 rad  theta_dot=+0.000 rad/s
t=10.00s  theta=+34598.655 rad  theta_dot=+0.000 rad/s


In [ ]:
from pydrake.systems.drawing import plot_system_graphviz


plot_system_graphviz(diagram)
plt.show()

FileNotFoundError: [Errno 2] "dot" not found in path.